# Homework 3

by: Najla Tsabita Afiyah

In [31]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [34]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [35]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna('NA')

for col in df.select_dtypes(include=['number']).columns:
    df[col] = df[col].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Q1. Mode for industry
What is the most frequent observation (mode) for the column industry?

In [36]:
df['industry'].mode()[0]

'technology'

### Q2. Biggest correlation
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

In [37]:
numerical_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
corr_matrix = df[numerical_cols].corr()
print(corr_matrix)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   
converted                      0.189652                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.122842    0.229496   0.189652  
number_of_courses_viewed           0.721609    0.757204   0.395038  
interaction_count                  1.000000    0.915746   0.448624  
lead_score                         0.915746    1.000000   0.483246  
converted                          0.448624    0.483246   1.000000  


#### Split the data

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

### Q3. Biggest MI 
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

In [40]:
from sklearn.metrics import mutual_info_score

In [41]:
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']

def calculate_mi(series):
    return mutual_info_score(series, df_full_train.converted)

mi_scores = df_full_train[categorical_cols].apply(calculate_mi)

mi_scores_rounded = mi_scores.apply(lambda x: round(x, 2)).sort_values(ascending=False)
print(mi_scores_rounded)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


### Q4. Accuracy 
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
- To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
- model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?